# Linear regression diagnostics

Fit a two-predictor OLS model on synthetic data and inspect fit quality, residuals and influence.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(5)
n = 120
x1 = rng.normal(50, 10, n)
x2 = rng.uniform(0, 5, n)
y = 2.0 + 0.4 * x1 - 3.2 * x2 + rng.normal(0, 6, n)
X = np.column_stack([np.ones(n), x1, x2])
beta, *_ = np.linalg.lstsq(X, y, rcond=None)
yhat = X @ beta
res = y - yhat
sigma2 = res @ res / (n - 3)
cov = sigma2 * np.linalg.inv(X.T @ X)
se = np.sqrt(np.diag(cov))
for i, name in enumerate(['intercept', 'x1', 'x2']):
    print(f'{name:9s} beta={beta[i]:+.3f} ± {se[i]:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
lo, hi = y.min(), y.max()
axes[0].scatter(yhat, y, s=14, alpha=0.6)
axes[0].plot([lo, hi], [lo, hi], color='#e05b5b', ls='--')
axes[0].set_xlabel('fitted'); axes[0].set_ylabel('observed')
axes[0].set_title('Observed vs fitted')
axes[1].scatter(yhat, res, s=14, alpha=0.6)
axes[1].axhline(0, color='#e05b5b', ls='--')
axes[1].set_xlabel('fitted'); axes[1].set_ylabel('residual')
axes[1].set_title('Residuals')

In [ ]:
H = X @ np.linalg.inv(X.T @ X) @ X.T
lev = np.diag(H)
cooks = res**2 / (3 * sigma2) * lev / (1 - lev)**2

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(range(n), lev, color='#4f8cff')
axes[0].axhline(2 * 3 / n, color='#e05b5b', ls='--', label='2p/n')
axes[0].set_title('Leverage'); axes[0].legend()
axes[1].bar(range(n), cooks, color='#d9a441')
axes[1].set_title("Cook's distance")
print(f'high-leverage points: {int((lev > 2*3/n).sum())}')